# E007 — Meta-Equilibrium, Conditional Reactions, and Probe Audit

This notebook converts the policy zoo from a brittle pure best-response system into a robust population strategy, and mines conditional reactions around market shocks. Active market probes remain **research-only** unless a paired simulator A/B clears the promotion gate.

## Kaggle input checklist
- **Input 1:** E001 artifacts containing `turns.parquet`
- **Input 2:** E002 artifacts containing `bt_strength.csv`
- **Input 3:** E005 policy-zoo evaluation containing `policy_matchups.parquet` or `.csv` with columns `policy`, `opponent_archetype`, `score`
- **Optional:** `policy_params.json` mapping policy names to ParametricMind parameters
- **Accelerator:** None
- **Internet:** Off
- **Outputs:** `reaction_events.parquet`, `reaction_profiles.parquet`, `reaction_archetypes.parquet`, `reaction_model.json`, `probe_thresholds.json`, `meta_artifact.json`


In [ ]:
from pathlib import Path
import json, sys
import pandas as pd

roots=[Path('/kaggle/input'),Path('/kaggle/working'),Path('.')]
repo=next((p for r in roots for p in r.rglob('src/kagv2/equilibrium.py')),None)
if repo is None: raise FileNotFoundError('Add the kaggriculture repository as an input')
ROOT=repo.parents[2]
sys.path.insert(0,str(ROOT/'src'));sys.path.insert(0,str(ROOT))
OUT=Path('/kaggle/working/kagv2');OUT.mkdir(parents=True,exist_ok=True)
from kagv2.replay import add_future_opponent_sell_labels
from kagv2.reactions import extract_reaction_events,reaction_profiles,fit_reaction_archetypes
from kagv2.probes import threshold_reactivity
from kagv2.equilibrium import payoff_from_results,robust_population_mix
print('repo',ROOT)


In [ ]:
def find_one(name):
    hits=[p for r in roots for p in r.rglob(name)]
    return hits[0] if hits else None
turn_path=find_one('turns.parquet');bt_path=find_one('bt_strength.csv')
if turn_path is None: raise FileNotFoundError('turns.parquet')
turns=pd.read_parquet(turn_path)
bt=pd.read_csv(bt_path) if bt_path else pd.DataFrame()
turns1=add_future_opponent_sell_labels(turns,horizon=1)
events=extract_reaction_events(turns1,price_threshold=.10,inventory_threshold=30,post_turns=12)
profiles=reaction_profiles(events,bt_strength=bt,min_events=3)
events.to_parquet(OUT/'reaction_events.parquet',index=False)
profiles.to_parquet(OUT/'reaction_profiles.parquet',index=False)
print('events',len(events),'profiles',len(profiles))
if len(profiles)>=2:
    clustered,rmodel=fit_reaction_archetypes(profiles,n_clusters=min(8,max(2,len(profiles)//8)))
    clustered.to_parquet(OUT/'reaction_archetypes.parquet',index=False)
    (OUT/'reaction_model.json').write_text(json.dumps(rmodel,indent=2))


In [ ]:
thresholds={}
for p in ['STRAWBERRY','MELON','MILK','WOOL','WHEAT','FERTILIZER']:
    t=threshold_reactivity(turns1,p,bins=24,min_bin=20)
    thresholds[p]=t.to_dict(orient='records')
(OUT/'probe_thresholds.json').write_text(json.dumps(thresholds,indent=2,default=str))
print('threshold tables written; these do NOT enable live probes')


In [ ]:
match_path=find_one('policy_matchups.parquet') or find_one('policy_matchups.csv')
if match_path is None:
    print('No policy_matchups file yet. Run E005 population evaluation first.')
else:
    m=pd.read_parquet(match_path) if match_path.suffix=='.parquet' else pd.read_csv(match_path)
    policies,opponents,A,N=payoff_from_results(m,shrink=8.0)
    counts=m.groupby('opponent_archetype').size().reindex(opponents,fill_value=0).to_numpy(float)
    prior=(counts/counts.sum()).tolist()
    meta=robust_population_mix(A,opponent_prior=prior,equilibrium_weight=.40,iterations=12000)
    params_path=find_one('policy_params.json');params=json.loads(params_path.read_text()) if params_path else {}
    meta.update({'policy_names':policies,'archetype_names':opponents,'payoff':A.tolist(),'match_counts':N.tolist(),'policy_params':params,'default_policy':policies[0] if policies else None,'min_archetype_confidence':.62,'switch_margin':.025,'prior_strength':.018})
    (OUT/'meta_artifact.json').write_text(json.dumps(meta,indent=2,sort_keys=True))
    print('mixture',dict(zip(policies,meta['policy_mixture'])))
    print('expected meta value',meta['expected_meta_value'],'worst archetype',meta['worst_archetype_value'],'duality gap',meta['equilibrium']['duality_gap'])
